In [4]:
import pandas as pd
import os
from datetime import date 
from rapidfuzz import process, fuzz

def norm_keys_to_int(df, keys):
    for k in keys:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors='coerce').astype('Int64')

fecha_hoy  = date.today().strftime("%Y.%m.%d")
ruta_sap   = r"C:\Users\jesus\GESTIONA ACTIVOS S.A.C\TGA_AAQ - 2402_PMO MEJORA ESTRATEGIAS - 2402_PMO MEJORA ESTRATEGIAS\03.01 EJECUCION TI\04. RQ TI Dev\00. Data\SAP\01 SAP C601 2025.06.02"
ruta_hdr   = r"IA09.xlsx"
ruta_pos   = r"IP18.xlsx"
ruta_op    = r"IA10_OP_Format.xlsx"
ruta_maf   = r"IA10_MAF_Format.xlsx"
ruta_maf_d = r"CV04N.xlsx"


HDR   = pd.read_excel(os.path.join(ruta_sap, ruta_hdr))
OP    = pd.read_excel(os.path.join(ruta_sap, ruta_op))
POS   = pd.read_excel(os.path.join(ruta_sap, ruta_pos))
MAF   = pd.read_excel(os.path.join(ruta_sap, ruta_maf))
MAF_D = pd.read_excel(os.path.join(ruta_sap, ruta_maf_d))

HDR = HDR.rename(columns={'Descripción': 'Cabecera HDR'}).drop(
                                                                columns=["Fe.modificación", "Centro", 
                                                                        "Creado el", "Creado por",  "Ind.borrado",
                                                                        "Modificado por", "Pet.borrado",  "In.validez",
                                                                        "Ubicac.técnica", "Ctro.pto.trab.", "HRuta jerárq.", "Elemento ref.PM/PS"], 
                                                                errors='ignore')


OP  = OP.rename(columns={'Grupo hojas ruta': 'Gpo.hojas ruta',
                         'Cont.grupo HRuta': 'Cont.gpo.HRuta'})
key_cols = ['Gpo.hojas ruta', 'Cont.gpo.HRuta']

for df in (HDR, POS, OP):
    norm_keys_to_int(df, key_cols)




In [10]:

HDR_OP = pd.merge(HDR, OP, on = ["Gpo.hojas ruta", "Cont.gpo.HRuta"], how ='inner')



In [11]:
HDR_OP = pd.merge(HDR_OP, POS[["Gpo.hojas ruta", "Cont.gpo.HRuta", "Ubicac.técnica"]], on = ["Gpo.hojas ruta", "Cont.gpo.HRuta"], how ='left')



In [12]:

HDR_OP_MAF = pd.merge(HDR_OP, MAF, left_on = ["Gpo.hojas ruta", "Cont.gpo.HRuta"], right_on = ["GRUPO", "CONTADOR"], how ='left')

HDR_OP_MAF = pd.merge(HDR_OP_MAF, MAF_D[["Documento","Document Long Text"]], left_on = ["DocumentoMAF"], right_on = ["Documento"], how ='left')

HDR_OP_MAF.to_excel(os.path.join(ruta_sap, f"HDR_OP_MAF_{fecha_hoy}.xlsx"), index = False )

In [ ]:

######################################################################################

ruta_prts_entregables = r"C:\Users\jesus\GESTIONA ACTIVOS S.A.C\TGA_AAQ - 2402_PMO MEJORA ESTRATEGIAS - 2402_PMO MEJORA ESTRATEGIAS\03.00 EJECUCION SEMANAL\03. Entregables\REQ_7"
# lista_equipos = os.listdir(ruta_prts_entregables)


ruta_data_input  = r"C:\Users\jesus\GESTIONA ACTIVOS S.A.C\TGA_AAQ - 2402_PMO MEJORA ESTRATEGIAS - 2402_PMO MEJORA ESTRATEGIAS\03.01 EJECUCION TI\04. RQ TI Dev\00. Data\ORIEN\250817"
ruta_data_output = r"" 

OPERACION_INPUT = "20258181448_Quellaveco_ExportacindeOperacionesdeAltoNivel_ompLpwSkK.xlsx"
TAREAS_INPUT    = "20258181514_Quellaveco_ExportacindeListadeTareasdeAltoNivel_eD9zAzzJb.xlsx"
EQUIPOS_INPUT   = "lista_equipos.xlsx"

df_OPERACION_INPUT = pd.read_excel(os.path.join(ruta_data_input, OPERACION_INPUT ), sheet_name="Operaciones de Alto Nivel",header=9)
df_TAREAS_INPUT    = pd.read_excel(os.path.join(ruta_data_input, TAREAS_INPUT ), sheet_name="Lista de Tareas de Alto Nivel",header=9)
df_EQUIPOS_INPUT   = pd.read_excel(os.path.join(ruta_data_input, EQUIPOS_INPUT ))

col_ubi = "Código de Ubicación"
col_op  = "Descripción de Operación"
df_OPERACION_INPUT.columns = (
    df_OPERACION_INPUT.columns.astype(str)
      .str.strip()
      .str.replace("\xa0", " ", regex=False)
)

df_OPERACION_INPUT = df_OPERACION_INPUT[[col_ubi, col_op, "Mano de Obra","No. de Personas Requeridas", "Trabajo"]]
df_TAREAS_INPUT    = df_TAREAS_INPUT[["Código de Ubicación de la Lista de Tareas", "Nombre de Lista de Tareas", "Estrategia de Mantenimiento"]]


df_ORIEN = pd.merge(df_OPERACION_INPUT, df_TAREAS_INPUT, left_on=col_ubi, right_on="Código de Ubicación de la Lista de Tareas",how="left" )
df_ORIEN = df_ORIEN[df_ORIEN[col_ubi].isin(df_EQUIPOS_INPUT["Equipos"])]


equipos = df_ORIEN[col_ubi].dropna().unique()
resultado = []  # filas por operación clasificada
umbral  = 75
print(HDR_OP_MAF[HDR_OP_MAF["Ubicac.técnica"] == "CH4000020"].drop_duplicates())

for eq in equipos:
    ops_new = df_ORIEN.loc[df_ORIEN[col_ubi] == eq, col_op].astype(str).tolist()
    ops_old = HDR_OP_MAF.loc[HDR_OP_MAF["Ubicac.técnica"] == eq, "Descripción"].astype(str).tolist()
    set_old = set(ops_old)
    set_new = set(ops_new)


    for op in ops_new:
        if op in set_old:
            resultado.append({
                col_ubi: eq,
                "op_nueva": op,
                "op_old": op,
                "score": 100,
                "estado": "SIN ACCIÓN"
            })
            continue

        match = process.extractOne(op, ops_old, scorer=fuzz.ratio)
        if not match or match[1] < umbral:
            resultado.append({
                col_ubi: eq,
                "op_nueva": op,
                "op_old": None,
                "score": 0,
                "estado": "nueva"
            })
        else:
            resultado.append({
                                col_ubi: eq,
                                "op_nueva": op,
                                "op_old": match[0],
                                "score": int(match[1]),
                                "estado": "actualizada"
                        })

        # eliminadas (opcionales)
        eliminadas = sorted(set_old - set_new)
        for op_old in eliminadas:
                resultado.append({
                                col_ubi: eq,
                                "op_nueva": None,
                                "op_old": op_old,
                                "score": 100,
                                "estado": "eliminada"
                                })
                
df_resultado = pd.DataFrame(resultado)
print(df_resultado[df_resultado[col_ubi] == "CH4000020"].drop_duplicates())
